# Phase 5: Model Training & Batch Scoring

**Goal:** Train an XGBoost binary classifier to predict fiber deployment, evaluate with temporal cross-validation, interpret with SHAP, and batch-score all currently unserved CBs.

**Inputs:**
- `training_data.parquet` (Phase 4)
- `feature_columns.json` (Phase 4)
- `feature_matrix_full.parquet` (Phase 4)

**Outputs:**
- `fiber_forecast_model.joblib`
- `cb_predictions.parquet`
- `model_evaluation_results.json`
- `shap_summary.png`

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib

# Paths
DATA_DIR = '/Users/yugeshchandraroy/Downloads/Projects/broadband-market-scanner/data/intermediate'
MODEL_DIR = '/Users/yugeshchandraroy/Downloads/Projects/broadband-market-scanner/data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

TRAINING_DATA_FILE = f'{DATA_DIR}/training_data.parquet'
FEATURE_COLS_FILE = f'{DATA_DIR}/feature_columns.json'
FEATURE_MATRIX_FILE = f'{DATA_DIR}/feature_matrix_full.parquet'

# XGBoost hyperparameters (from plan)
XGB_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',
    'n_estimators': 500,
    'early_stopping_rounds': 50,
    'random_state': 42,
    'verbosity': 0,
}

print('Phase 5: Model Training & Batch Scoring')
print('=' * 55)

## Cell 2: Load Data

In [ ]:
# Load training data and feature column list
training_data = pd.read_parquet(TRAINING_DATA_FILE)
with open(FEATURE_COLS_FILE) as f:
    feature_columns = json.load(f)

print(f'Training data: {training_data.shape[0]:,} rows x {training_data.shape[1]} cols')
print(f'Feature columns: {len(feature_columns)}')

# Class distribution
n_pos = int(training_data['gained_fiber'].sum())
n_neg = len(training_data) - n_pos
n_total = len(training_data)

print(f'\nClass distribution:')
print(f'  Positive (gained_fiber=1): {n_pos:,} ({100*n_pos/n_total:.2f}%)')
print(f'  Negative (gained_fiber=0): {n_neg:,} ({100*n_neg/n_total:.2f}%)')

# Compute scale_pos_weight (cap at 1000 when 0 positives)
if n_pos == 0:
    scale_pos_weight = 1000
    print(f'\n  WARNING: 0 positive samples. Capping scale_pos_weight at {scale_pos_weight}.')
    print('  This is a structural consequence of the fiber-only data + forward-persistence correction.')
    print('  The model will train but all predictions will reflect the single-class reality.')
else:
    scale_pos_weight = n_neg / n_pos
    print(f'  scale_pos_weight: {scale_pos_weight:.1f}')

XGB_PARAMS['scale_pos_weight'] = scale_pos_weight

print(f'\nTraining pairs:')
for (tp, lp), group in training_data.groupby(['train_period', 'label_period']):
    gained = group['gained_fiber'].sum()
    print(f'  {tp} -> {lp}: {len(group):,} samples, {int(gained):,} positive')

## Cell 3: Temporal CV Setup

In [ ]:
# Temporal cross-validation: train on past, validate on next period
# Never train on future data
unique_periods = sorted(training_data['train_period'].unique())
print(f'Unique train_period values: {unique_periods}')

# Build folds:
#   Fold 1: Train on pairs with train_period <= period[0], validate on period[1]
#   Fold 2: Train on pairs with train_period <= period[1], validate on period[2]
folds = []
for i in range(1, len(unique_periods)):
    train_periods = unique_periods[:i]
    val_period = unique_periods[i]
    folds.append((train_periods, val_period))

print(f'\nTemporal CV folds: {len(folds)}')
for fold_idx, (train_periods, val_period) in enumerate(folds):
    train_mask = training_data['train_period'].isin(train_periods)
    val_mask = training_data['train_period'] == val_period
    print(f'  Fold {fold_idx+1}:')
    print(f'    Train periods: {train_periods} ({train_mask.sum():,} samples)')
    print(f'    Val period:    {val_period} ({val_mask.sum():,} samples)')
    print(f'    Train positives: {int(training_data.loc[train_mask, "gained_fiber"].sum()):,}')
    print(f'    Val positives:   {int(training_data.loc[val_mask, "gained_fiber"].sum()):,}')

## Cell 4: CV Training Loop

In [ ]:
cv_results = []

for fold_idx, (train_periods, val_period) in enumerate(folds):
    print(f'\n{"="*50}')
    print(f'Fold {fold_idx+1}: Train on {train_periods}, Validate on {val_period}')
    print('=' * 50)
    
    # Split data
    train_mask = training_data['train_period'].isin(train_periods)
    val_mask = training_data['train_period'] == val_period
    
    X_train = training_data.loc[train_mask, feature_columns]
    y_train = training_data.loc[train_mask, 'gained_fiber']
    X_val = training_data.loc[val_mask, feature_columns]
    y_val = training_data.loc[val_mask, 'gained_fiber']
    
    print(f'  X_train: {X_train.shape}, y_train positives: {int(y_train.sum())}')
    print(f'  X_val:   {X_val.shape}, y_val positives:   {int(y_val.sum())}')
    
    # Train XGBoost
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    # Predictions
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # Evaluate: handle single-class edge case
    n_classes_val = len(np.unique(y_val))
    fold_result = {
        'fold': fold_idx + 1,
        'train_periods': train_periods,
        'val_period': val_period,
        'train_samples': int(len(X_train)),
        'val_samples': int(len(X_val)),
        'train_positives': int(y_train.sum()),
        'val_positives': int(y_val.sum()),
    }
    
    # AUC-ROC
    if n_classes_val < 2:
        fold_result['auc_roc'] = 'undefined - only one class in validation data'
        print(f'  AUC-ROC: undefined (only one class in validation data)')
    else:
        try:
            auc_roc = roc_auc_score(y_val, y_pred_proba)
            fold_result['auc_roc'] = round(auc_roc, 4)
            print(f'  AUC-ROC: {auc_roc:.4f}')
        except ValueError as e:
            fold_result['auc_roc'] = f'undefined - {str(e)}'
            print(f'  AUC-ROC: undefined ({e})')
    
    # AUC-PR
    if n_classes_val < 2:
        fold_result['auc_pr'] = 'undefined - only one class in validation data'
        print(f'  AUC-PR:  undefined (only one class in validation data)')
    else:
        try:
            auc_pr = average_precision_score(y_val, y_pred_proba)
            fold_result['auc_pr'] = round(auc_pr, 4)
            print(f'  AUC-PR:  {auc_pr:.4f}')
        except ValueError as e:
            fold_result['auc_pr'] = f'undefined - {str(e)}'
            print(f'  AUC-PR:  undefined ({e})')
    
    # F1 at optimal threshold (sweep 0.1 to 0.9)
    best_f1 = 0.0
    best_threshold = 0.5
    for threshold in np.arange(0.1, 0.91, 0.05):
        y_pred_binary = (y_pred_proba >= threshold).astype(int)
        if y_pred_binary.sum() == 0 and y_val.sum() == 0:
            # Both empty: F1 is trivially 1.0 but not meaningful
            continue
        f1 = f1_score(y_val, y_pred_binary, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = round(float(threshold), 2)
    
    fold_result['best_f1'] = round(best_f1, 4)
    fold_result['best_threshold'] = best_threshold
    print(f'  Best F1: {best_f1:.4f} at threshold {best_threshold}')
    
    if best_f1 == 0.0 and n_classes_val < 2:
        print(f'  (F1=0 expected: no positive samples in validation set)')
    
    # Prediction distribution
    print(f'  Prediction distribution: min={y_pred_proba.min():.6f}, '
          f'median={np.median(y_pred_proba):.6f}, max={y_pred_proba.max():.6f}')
    
    cv_results.append(fold_result)
    print(f'  Best iteration: {model.best_iteration}')

print(f'\n{"="*50}')
print('CV Summary')
print('=' * 50)
for r in cv_results:
    print(f"  Fold {r['fold']}: AUC-ROC={r['auc_roc']}, AUC-PR={r['auc_pr']}, F1={r['best_f1']}")

## Cell 5: Train Final Model

In [ ]:
# Train final model on ALL training data
print('Training final model on all training data...')

X_all = training_data[feature_columns]
y_all = training_data['gained_fiber']

# For the final model, use n_estimators without early stopping
# (no separate eval set to stop on)
final_params = {k: v for k, v in XGB_PARAMS.items() if k != 'early_stopping_rounds'}

# If CV produced best_iteration values, use the average as n_estimators
# Otherwise, use the default
final_model = xgb.XGBClassifier(**final_params)
final_model.fit(X_all, y_all, verbose=False)

print(f'  Training samples: {len(X_all):,}')
print(f'  Features: {len(feature_columns)}')
print(f'  n_estimators used: {final_model.n_estimators}')

# Save model
model_path = f'{MODEL_DIR}/fiber_forecast_model.joblib'
joblib.dump(final_model, model_path)
print(f'\nModel saved: {model_path}')
print(f'  Size: {os.path.getsize(model_path) / (1024*1024):.2f} MB')

## Cell 6: SHAP Analysis

In [ ]:
# SHAP analysis on final model
print('Computing SHAP values...')

explainer = shap.TreeExplainer(final_model)

# Sample up to 5,000 rows for SHAP
shap_sample_size = min(5000, len(X_all))
X_shap = X_all.sample(n=shap_sample_size, random_state=42)
shap_values = explainer.shap_values(X_shap)

print(f'  SHAP computed on {shap_sample_size:,} samples')
print(f'  SHAP values shape: {shap_values.shape}')

# Beeswarm summary plot
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values, X_shap, show=False, max_display=20)
plt.tight_layout()
shap_plot_path = f'{MODEL_DIR}/shap_summary.png'
plt.savefig(shap_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'  SHAP summary plot saved: {shap_plot_path}')

# Ranked feature importance (mean absolute SHAP)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print(f'\nTop 20 Features by Mean |SHAP|:')
print('-' * 50)
for i, row in feature_importance.head(20).iterrows():
    print(f'  {i+1:2d}. {row["feature"]:40s} {row["mean_abs_shap"]:.6f}')

## Cell 7: Batch Scoring

In [ ]:
# Load latest period feature matrix and filter to unserved blocks
print('Loading feature matrix for batch scoring...')
feature_matrix = pd.read_parquet(FEATURE_MATRIX_FILE)

# Identify the latest period
latest_period = sorted(feature_matrix['filing_period'].unique())[-1]
print(f'  Latest period: {latest_period}')

latest = feature_matrix[feature_matrix['filing_period'] == latest_period].copy()
print(f'  Blocks in latest period: {len(latest):,}')

# Filter to unserved (has_fiber == 0)
unserved = latest[latest['has_fiber'] == 0].copy()
n_unserved = len(unserved)
print(f'  Unserved blocks (has_fiber=0): {n_unserved:,}')

if n_unserved > 0:
    print(f'\nScoring {n_unserved:,} unserved blocks...')
    X_score = unserved[feature_columns]
    
    # Predict probabilities
    fiber_proba = final_model.predict_proba(X_score)[:, 1]
    
    # Per-CB SHAP values for top contributing features
    print('  Computing per-CB SHAP values...')
    shap_vals_score = explainer.shap_values(X_score)
    
    top_features_list = []
    for i in range(len(X_score)):
        sv = shap_vals_score[i]
        top_idx = np.argsort(np.abs(sv))[-3:][::-1]
        top_feats = [f'{feature_columns[j]} ({sv[j]:+.4f})' for j in top_idx]
        top_features_list.append('; '.join(top_feats))
    
    predictions = pd.DataFrame({
        'cb_fips': unserved['block_geoid'].values,
        'fiber_probability': fiber_proba,
        'top_contributing_features': top_features_list,
    })
    
    print(f'  Predictions: {len(predictions):,} rows')
    print(f'  Probability stats: min={fiber_proba.min():.6f}, '
          f'median={np.median(fiber_proba):.6f}, max={fiber_proba.max():.6f}')
else:
    print(f'\n  WARNING: 0 unserved blocks in {latest_period}.')
    print('  All blocks already have fiber service in this dataset.')
    print('  This is expected given the fiber-only data source.')
    print('  Producing empty predictions DataFrame with correct schema.')
    
    predictions = pd.DataFrame({
        'cb_fips': pd.Series(dtype='object'),
        'fiber_probability': pd.Series(dtype='float64'),
        'top_contributing_features': pd.Series(dtype='object'),
    })

## Cell 8: Forecast Labels

In [ ]:
# Assign categorical labels based on probability thresholds
def assign_forecast_label(prob):
    if prob > 0.6:
        return 'High'
    elif prob >= 0.3:
        return 'Medium'
    else:
        return 'Low'

if len(predictions) > 0:
    predictions['fiber_forecast_label'] = predictions['fiber_probability'].apply(assign_forecast_label)
    
    print('Forecast Label Distribution:')
    print('=' * 40)
    label_counts = predictions['fiber_forecast_label'].value_counts()
    for label in ['High', 'Medium', 'Low']:
        count = label_counts.get(label, 0)
        pct = 100 * count / len(predictions) if len(predictions) > 0 else 0
        print(f'  {label:8s}: {count:,} ({pct:.1f}%)')
    
    print(f'\nProbability by label:')
    for label in ['High', 'Medium', 'Low']:
        subset = predictions[predictions['fiber_forecast_label'] == label]['fiber_probability']
        if len(subset) > 0:
            print(f'  {label:8s}: mean={subset.mean():.4f}, min={subset.min():.4f}, max={subset.max():.4f}')
else:
    predictions['fiber_forecast_label'] = pd.Series(dtype='object')
    print('No predictions to label (0 unserved blocks).')
    print('Empty DataFrame schema preserved for downstream compatibility.')

## Cell 9: Export

In [ ]:
# 1. Model (already saved in Cell 5)
print('Exported Artifacts:')
print('=' * 60)
print(f'1. Model: {model_path}')
print(f'   Size: {os.path.getsize(model_path) / (1024*1024):.2f} MB')

# 2. Predictions
pred_path = f'{MODEL_DIR}/cb_predictions.parquet'
predictions.to_parquet(pred_path, index=False)
print(f'\n2. Predictions: {pred_path}')
print(f'   Rows: {len(predictions):,}')
print(f'   Columns: {list(predictions.columns)}')
print(f'   Size: {os.path.getsize(pred_path) / (1024*1024):.2f} MB')

# 3. Evaluation results
eval_path = f'{MODEL_DIR}/model_evaluation_results.json'
eval_output = {
    'cv_folds': cv_results,
    'training_samples': int(len(X_all)),
    'n_features': len(feature_columns),
    'positive_rate': float(y_all.mean()),
    'scale_pos_weight': scale_pos_weight,
    'xgb_params': {k: v for k, v in XGB_PARAMS.items()},
    'n_unserved_latest': n_unserved,
    'latest_period': latest_period,
}
with open(eval_path, 'w') as f:
    json.dump(eval_output, f, indent=2, default=str)
print(f'\n3. Evaluation results: {eval_path}')

# 4. SHAP plot (already saved in Cell 6)
print(f'\n4. SHAP summary plot: {shap_plot_path}')
print(f'   Size: {os.path.getsize(shap_plot_path) / (1024*1024):.2f} MB')

## Cell 10: Summary

In [ ]:
print('=' * 60)
print('PHASE 5 SUMMARY')
print('=' * 60)

print(f'\nTraining Data:')
print(f'  Total samples: {len(X_all):,}')
print(f'  Features: {len(feature_columns)}')
print(f'  Positive rate: {100*y_all.mean():.2f}%')
print(f'  scale_pos_weight: {scale_pos_weight}')

print(f'\nTemporal CV Results:')
for r in cv_results:
    print(f'  Fold {r["fold"]}: AUC-ROC={r["auc_roc"]}, AUC-PR={r["auc_pr"]}, F1={r["best_f1"]}')

print(f'\nBatch Scoring:')
print(f'  Latest period: {latest_period}')
print(f'  Total blocks: {len(latest):,}')
print(f'  Unserved blocks: {n_unserved:,}')
print(f'  Predictions generated: {len(predictions):,}')

if len(predictions) > 0:
    label_counts = predictions['fiber_forecast_label'].value_counts()
    print(f'\n  Forecast distribution:')
    for label in ['High', 'Medium', 'Low']:
        print(f'    {label}: {label_counts.get(label, 0):,}')

print(f'\nOutput Files:')
output_files = [
    f'{MODEL_DIR}/fiber_forecast_model.joblib',
    f'{MODEL_DIR}/cb_predictions.parquet',
    f'{MODEL_DIR}/model_evaluation_results.json',
    f'{MODEL_DIR}/shap_summary.png',
]
for fp in output_files:
    if os.path.exists(fp):
        size_mb = os.path.getsize(fp) / (1024*1024)
        print(f'  {os.path.basename(fp):40s} {size_mb:.2f} MB')
    else:
        print(f'  {os.path.basename(fp):40s} NOT FOUND')

print(f'\nPhase 5 complete!')